In [46]:
import pandas as pd

In [47]:
df = pd.read_excel("three_phase_IM_random_dataset.xlsx")
df.head()

,Torque_Nm,Flux_Wb,Angle_deg,Sector,Pulse_A_Upper,Pulse_B_Upper,Pulse_C_Upper,Pulse_A_Lower,Pulse_B_Lower,Pulse_C_Lower
0,20.867,1.046,211.048,4,0,1,0,1,1,0
1,26.941,0.638,132.754,3,0,1,0,1,0,1
2,0.745,0.723,224.578,4,0,1,0,1,1,0
3,1.956,0.161,154.833,3,0,1,0,1,0,1
4,0.522,0.269,299.427,5,0,1,1,0,1,0


In [48]:
df1 = df.drop(["Angle_deg"] , axis = 1)
df1.head()

,Torque_Nm,Flux_Wb,Sector,Pulse_A_Upper,Pulse_B_Upper,Pulse_C_Upper,Pulse_A_Lower,Pulse_B_Lower,Pulse_C_Lower
0,20.867,1.046,4,0,1,0,1,1,0
1,26.941,0.638,3,0,1,0,1,0,1
2,0.745,0.723,4,0,1,0,1,1,0
3,1.956,0.161,3,0,1,0,1,0,1
4,0.522,0.269,5,0,1,1,0,1,0


In [49]:
df2 = df.drop(["Sector"] , axis = 1)
df2.head()

,Torque_Nm,Flux_Wb,Angle_deg,Pulse_A_Upper,Pulse_B_Upper,Pulse_C_Upper,Pulse_A_Lower,Pulse_B_Lower,Pulse_C_Lower
0,20.867,1.046,211.048,0,1,0,1,1,0
1,26.941,0.638,132.754,0,1,0,1,0,1
2,0.745,0.723,224.578,0,1,0,1,1,0
3,1.956,0.161,154.833,0,1,0,1,0,1
4,0.522,0.269,299.427,0,1,1,0,1,0


## df1 we removed angle and in df2 we removed sector

## working with df1

In [50]:
import torch

In [51]:
import torch.nn as nn

In [52]:
class MotorModel_1(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(4 , 64)
        self.l2 = nn.Linear(64 , 128)
        self.l3 = nn.Linear(128 , 64)
        self.l4 = nn.Linear(64 , 6)
        self.r = nn.ReLU()
        self.d = nn.Dropout(0.15)
    def forward(self , x):
        return self.l4(self.d(self.r(self.l3(self.d(self.r(self.l2(self.d(self.r(self.l1(x))))))))))

In [53]:
df1.dtypes

Torque_Nm        float64
Flux_Wb          float64
Sector             int64
Pulse_A_Upper      int64
Pulse_B_Upper      int64
Pulse_C_Upper      int64
Pulse_A_Lower      int64
Pulse_B_Lower      int64
Pulse_C_Lower      int64
dtype: object

In [54]:
df1.head()

,Torque_Nm,Flux_Wb,Sector,Pulse_A_Upper,Pulse_B_Upper,Pulse_C_Upper,Pulse_A_Lower,Pulse_B_Lower,Pulse_C_Lower
0,20.867,1.046,4,0,1,0,1,1,0
1,26.941,0.638,3,0,1,0,1,0,1
2,0.745,0.723,4,0,1,0,1,1,0
3,1.956,0.161,3,0,1,0,1,0,1
4,0.522,0.269,5,0,1,1,0,1,0


In [55]:
x1 = df1.drop(["Pulse_A_Upper"	, "Pulse_B_Upper" , "Pulse_C_Upper" ,	"Pulse_A_Lower" ,	"Pulse_B_Lower" ,	"Pulse_C_Lower"] , axis = 1)
x1.head()

,Torque_Nm,Flux_Wb,Sector
0,20.867,1.046,4
1,26.941,0.638,3
2,0.745,0.723,4
3,1.956,0.161,3
4,0.522,0.269,5


In [56]:
y1 =  df1.drop(["Torque_Nm"	, "Flux_Wb" ,	"Sector"] , axis = 1)
y1.head()

,Pulse_A_Upper,Pulse_B_Upper,Pulse_C_Upper,Pulse_A_Lower,Pulse_B_Lower,Pulse_C_Lower
0,0,1,0,1,1,0
1,0,1,0,1,0,1
2,0,1,0,1,1,0
3,0,1,0,1,0,1
4,0,1,1,0,1,0


In [57]:
import numpy as np
x1["Sector_sin"] = np.sin(2 * np.pi * df["Sector"] / 6)
x1["Sector_cos"] = np.cos(2 * np.pi * df["Sector"] / 6)

In [58]:
x1.head()

,Torque_Nm,Flux_Wb,Sector,Sector_sin,Sector_cos
0,20.867,1.046,4,-8.660254e-01,-0.5
1,26.941,0.638,3,1.224647e-16,-1.0
2,0.745,0.723,4,-8.660254e-01,-0.5
3,1.956,0.161,3,1.224647e-16,-1.0
4,0.522,0.269,5,-8.660254e-01,0.5


In [59]:
x1 = x1.drop(["Sector"], axis=1)
x1.head()

,Torque_Nm,Flux_Wb,Sector_sin,Sector_cos
0,20.867,1.046,-8.660254e-01,-0.5
1,26.941,0.638,1.224647e-16,-1.0
2,0.745,0.723,-8.660254e-01,-0.5
3,1.956,0.161,1.224647e-16,-1.0
4,0.522,0.269,-8.660254e-01,0.5


In [60]:
x1["Torque_Nm"].value_counts()

Torque_Nm
36.951    2
47.257    2
2.045     2
32.901    2
14.745    2
         ..
30.124    1
35.923    1
14.574    1
7.068     1
45.171    1
Name: count, Length: 988, dtype: int64

In [61]:
from sklearn.model_selection import train_test_split
x1_train , x1_test , y1_train , y1_test = train_test_split(x1 , y1 , random_state = 19 , test_size = 0.3)

In [62]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x1_train = scaler.fit_transform(x1_train)

x1_test = scaler.transform(x1_test)

In [63]:
x1_train = torch.tensor(x1_train, dtype=torch.float32)
x1_test = torch.tensor(x1_test, dtype=torch.float32)
y1_train = torch.tensor(y1_train.values, dtype=torch.float32)
y1_test = torch.tensor(y1_test.values, dtype=torch.float32)

In [64]:
x1_train

tensor([[-1.2235,  0.5334, -1.3194, -0.6470],
        [ 1.0705, -1.0283,  1.1734, -0.6470],
        [-1.5120,  1.5880, -1.3194, -0.6470],
        ...,
        [-0.1250,  0.0107,  1.1734,  0.7487],
        [-0.9330,  0.6796,  1.1734, -0.6470],
        [-1.1875, -0.7701, -1.3194, -0.6470]])

In [65]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [66]:
m1 = MotorModel_1().to(device)

In [67]:
loss = nn.BCEWithLogitsLoss()

In [68]:
import torch.optim as optim
optimizer = optim.Adam(m1.parameters() , lr = 0.001)

In [69]:
!pip install tqdm

In [70]:
def train_step(model , epoch , Acc , loss , optimizer , x1_train , y1_train , x1_test , y1_test):
    model = model.to(device)
    x1_train , y1_train , x1_test , y1_test = x1_train.to(device) , y1_train.to(device) , x1_test.to(device) , y1_test.to(device)
    loss = loss.to(device)
    from tqdm.auto import tqdm
    for i in tqdm(range(epoch)):
        model.train()
        y_pred = model(x1_train)
        train_loss = loss(y_pred, y1_train)
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        
        model.eval()
        with torch.inference_mode():
            test_pred = model(x1_test)
            test_loss = loss(test_pred, y1_test)
            pred_labels = (
                torch.sigmoid(test_pred) > 0.5
            ).float()
            acc = Acc(pred_labels, y1_test)

        if i % 2 == 0:

            print(f"""Epoch : {i}  Train Loss : {train_loss:.4f}  Test Loss : {test_loss:.4f}  Accuracy : {acc:.4f} """)

In [71]:
from torchmetrics.classification import MultilabelAccuracy
acc = MultilabelAccuracy(num_labels = 6).to(device)

In [72]:
!pip install torchmetrics

In [73]:
train_step(m1 ,100 , acc , loss , optimizer , x1_train , y1_train , x1_test , y1_test)

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.6965  Test Loss : 0.6910  Accuracy : 0.4994 
Epoch : 2  Train Loss : 0.6878  Test Loss : 0.6829  Accuracy : 0.5594 
Epoch : 4  Train Loss : 0.6805  Test Loss : 0.6744  Accuracy : 0.6000 
Epoch : 6  Train Loss : 0.6716  Test Loss : 0.6651  Accuracy : 0.6906 
Epoch : 8  Train Loss : 0.6623  Test Loss : 0.6543  Accuracy : 0.7972 
Epoch : 10  Train Loss : 0.6504  Test Loss : 0.6414  Accuracy : 0.8261 
Epoch : 12  Train Loss : 0.6373  Test Loss : 0.6258  Accuracy : 0.8456 
Epoch : 14  Train Loss : 0.6209  Test Loss : 0.6071  Accuracy : 0.8628 
Epoch : 16  Train Loss : 0.6006  Test Loss : 0.5848  Accuracy : 0.8794 
Epoch : 18  Train Loss : 0.5769  Test Loss : 0.5586  Accuracy : 0.9078 
Epoch : 20  Train Loss : 0.5487  Test Loss : 0.5283  Accuracy : 0.9333 
Epoch : 22  Train Loss : 0.5197  Test Loss : 0.4940  Accuracy : 0.9506 
Epoch : 24  Train Loss : 0.4829  Test Loss : 0.4560  Accuracy : 0.9639 
Epoch : 26  Train Loss : 0.4441  Test Loss : 0.4149  Accuracy : 0.965

In [76]:

class ResidualBlock(nn.Module):

    def __init__(self, features):

        super().__init__()

        self.layer = nn.Sequential(

            nn.Linear(features, features),
            nn.ReLU(),

            nn.Linear(features, features)

        )

        self.relu = nn.ReLU()

    def forward(self, x):

        identity = x

        out = self.layer(x)

        out = out + identity

        out = self.relu(out)

        return out

In [77]:
class DeepResNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.input = nn.Linear(4, 64)

        self.block1 = ResidualBlock(64)

        self.block2 = ResidualBlock(64)

        self.output = nn.Linear(64, 6)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.relu(self.input(x))

        x = self.block1(x)

        x = self.block2(x)

        x = self.output(x)

        return x

In [78]:
m2 = DeepResNet().to(device)

In [79]:
op = optim.Adam(m2.parameters() , lr = 0.001)

In [80]:
train_step(m2 ,100 , acc , loss , op , x1_train , y1_train , x1_test , y1_test)

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.7081  Test Loss : 0.6937  Accuracy : 0.5478 
Epoch : 2  Train Loss : 0.6863  Test Loss : 0.6723  Accuracy : 0.6033 
Epoch : 4  Train Loss : 0.6654  Test Loss : 0.6514  Accuracy : 0.6661 
Epoch : 6  Train Loss : 0.6448  Test Loss : 0.6306  Accuracy : 0.7044 
Epoch : 8  Train Loss : 0.6239  Test Loss : 0.6092  Accuracy : 0.7433 
Epoch : 10  Train Loss : 0.6021  Test Loss : 0.5867  Accuracy : 0.7961 
Epoch : 12  Train Loss : 0.5791  Test Loss : 0.5627  Accuracy : 0.8422 
Epoch : 14  Train Loss : 0.5543  Test Loss : 0.5368  Accuracy : 0.8756 
Epoch : 16  Train Loss : 0.5273  Test Loss : 0.5085  Accuracy : 0.8950 
Epoch : 18  Train Loss : 0.4976  Test Loss : 0.4775  Accuracy : 0.9228 
Epoch : 20  Train Loss : 0.4648  Test Loss : 0.4434  Accuracy : 0.9439 
Epoch : 22  Train Loss : 0.4289  Test Loss : 0.4063  Accuracy : 0.9656 
Epoch : 24  Train Loss : 0.3900  Test Loss : 0.3666  Accuracy : 0.9772 
Epoch : 26  Train Loss : 0.3485  Test Loss : 0.3245  Accuracy : 0.983

In [81]:
!pip install lightgbm

In [82]:
import torch
from torch import nn

class MLP(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.layer_stack = nn.Sequential(
            
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 6)
        )
        
    def forward(self, x):
        
        return self.layer_stack(x)


m1 = MLP()

In [83]:
op1 = optim.Adam(m1.parameters() , lr = 0.001)

In [84]:
train_step(m1 ,100 , acc , loss , op1 , x1_train , y1_train , x1_test , y1_test)

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.7011  Test Loss : 0.6981  Accuracy : 0.4944 
Epoch : 2  Train Loss : 0.6926  Test Loss : 0.6894  Accuracy : 0.5228 
Epoch : 4  Train Loss : 0.6831  Test Loss : 0.6804  Accuracy : 0.5589 
Epoch : 6  Train Loss : 0.6756  Test Loss : 0.6712  Accuracy : 0.6083 
Epoch : 8  Train Loss : 0.6653  Test Loss : 0.6616  Accuracy : 0.6689 
Epoch : 10  Train Loss : 0.6569  Test Loss : 0.6514  Accuracy : 0.7189 
Epoch : 12  Train Loss : 0.6446  Test Loss : 0.6407  Accuracy : 0.7661 
Epoch : 14  Train Loss : 0.6346  Test Loss : 0.6292  Accuracy : 0.8061 
Epoch : 16  Train Loss : 0.6235  Test Loss : 0.6169  Accuracy : 0.8400 
Epoch : 18  Train Loss : 0.6112  Test Loss : 0.6036  Accuracy : 0.8744 
Epoch : 20  Train Loss : 0.5977  Test Loss : 0.5893  Accuracy : 0.8961 
Epoch : 22  Train Loss : 0.5852  Test Loss : 0.5740  Accuracy : 0.9094 
Epoch : 24  Train Loss : 0.5692  Test Loss : 0.5577  Accuracy : 0.9211 
Epoch : 26  Train Loss : 0.5522  Test Loss : 0.5405  Accuracy : 0.937

In [85]:
import torch
from torch import nn

class MotorANN(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        self.network = nn.Sequential(
            
            nn.Linear(4, 50),
            nn.Sigmoid(),
            
            nn.Linear(50, 50),
            nn.Sigmoid(),
            
            nn.Linear(50, 6)
        )
        
    def forward(self, x):
        
        return self.network(x)


model = MotorANN()

In [86]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [87]:
train_step(model ,10 , acc , loss , optimizer , x1_train , y1_train , x1_test , y1_test)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.7214  Test Loss : 0.7167  Accuracy : 0.5000 
Epoch : 2  Train Loss : 0.7147  Test Loss : 0.7107  Accuracy : 0.5000 
Epoch : 4  Train Loss : 0.7089  Test Loss : 0.7055  Accuracy : 0.5000 
Epoch : 6  Train Loss : 0.7040  Test Loss : 0.7012  Accuracy : 0.5000 
Epoch : 8  Train Loss : 0.6998  Test Loss : 0.6975  Accuracy : 0.5000 


In [88]:
x1_train[0]

tensor([-1.2235,  0.5334, -1.3194, -0.6470])

In [89]:
import torch
from torch import nn

class ModernANN(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        self.network = nn.Sequential(
            
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 6)
        )
        
    def forward(self, x):
        
        return self.network(x)


modern_model = ModernANN()


In [90]:
modern_optimizer = torch.optim.Adam(
    modern_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [91]:
train_step(modern_model ,500 , acc , loss , modern_optimizer , x1_train , y1_train , x1_test , y1_test)

  0%|          | 0/500 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.6995  Test Loss : 0.6947  Accuracy : 0.5539 
Epoch : 2  Train Loss : 0.6910  Test Loss : 0.6864  Accuracy : 0.5889 
Epoch : 4  Train Loss : 0.6835  Test Loss : 0.6784  Accuracy : 0.6556 
Epoch : 6  Train Loss : 0.6761  Test Loss : 0.6703  Accuracy : 0.6989 
Epoch : 8  Train Loss : 0.6676  Test Loss : 0.6621  Accuracy : 0.7278 
Epoch : 10  Train Loss : 0.6592  Test Loss : 0.6536  Accuracy : 0.7472 
Epoch : 12  Train Loss : 0.6520  Test Loss : 0.6447  Accuracy : 0.7700 
Epoch : 14  Train Loss : 0.6421  Test Loss : 0.6353  Accuracy : 0.7817 
Epoch : 16  Train Loss : 0.6314  Test Loss : 0.6252  Accuracy : 0.7928 
Epoch : 18  Train Loss : 0.6238  Test Loss : 0.6143  Accuracy : 0.7978 
Epoch : 20  Train Loss : 0.6110  Test Loss : 0.6027  Accuracy : 0.8028 
Epoch : 22  Train Loss : 0.5985  Test Loss : 0.5901  Accuracy : 0.8100 
Epoch : 24  Train Loss : 0.5867  Test Loss : 0.5766  Accuracy : 0.8139 
Epoch : 26  Train Loss : 0.5739  Test Loss : 0.5622  Accuracy : 0.827

## now using df2

In [92]:
x2 = df2.drop(["Pulse_A_Upper"	, "Pulse_B_Upper" , "Pulse_C_Upper" ,	"Pulse_A_Lower" ,	"Pulse_B_Lower" ,	"Pulse_C_Lower"] , axis = 1)
x2.head()

,Torque_Nm,Flux_Wb,Angle_deg
0,20.867,1.046,211.048
1,26.941,0.638,132.754
2,0.745,0.723,224.578
3,1.956,0.161,154.833
4,0.522,0.269,299.427


In [93]:
y2 =  df2.drop(["Torque_Nm"	, "Flux_Wb" ,	"Angle_deg"] , axis = 1)
y2.head()

,Pulse_A_Upper,Pulse_B_Upper,Pulse_C_Upper,Pulse_A_Lower,Pulse_B_Lower,Pulse_C_Lower
0,0,1,0,1,1,0
1,0,1,0,1,0,1
2,0,1,0,1,1,0
3,0,1,0,1,0,1
4,0,1,1,0,1,0


In [94]:
x2["Angle_sin"] = np.sin(np.radians(x2["Angle_deg"]))
x2["Angle_cos"] = np.cos(np.radians(x2["Angle_deg"]))

In [95]:
x2.head()

,Torque_Nm,Flux_Wb,Angle_deg,Angle_sin,Angle_cos
0,20.867,1.046,211.048,-0.515756,-0.856736
1,26.941,0.638,132.754,0.734275,-0.678852
2,0.745,0.723,224.578,-0.701880,-0.712296
3,1.956,0.161,154.833,0.425258,-0.905072
4,0.522,0.269,299.427,-0.870982,0.491314


In [96]:
x2 = x2.drop(["Angle_deg"] , axis = 1)

In [97]:
from sklearn.model_selection import train_test_split
x2_train , x2_test , y2_train , y2_test = train_test_split(x2 , y2 , random_state = 1 , test_size = 0.3)

In [98]:
scaler = StandardScaler()
x2_train = scaler.fit_transform(x2_train)
x2_test = scaler.transform(x2_test)

In [101]:
import joblib
import os
os.mkdir("saved_models")
joblib.dump(scaler, "saved_models/scaler.pkl")

['saved_models/scaler.pkl']

In [102]:
x2_train = torch.tensor(x2_train, dtype=torch.float32)
x2_test = torch.tensor(x2_test, dtype=torch.float32)
y2_train = torch.tensor(y2_train.values, dtype=torch.float32)
y2_test = torch.tensor(y2_test.values, dtype=torch.float32)

In [103]:
x2_train

tensor([[-0.9327, -1.3734, -0.9611, -1.0985],
        [ 1.4968, -1.1011,  1.0341, -0.9007],
        [ 1.5707, -1.1899, -1.2114,  0.8315],
        ...,
        [-0.0547,  0.6087,  1.3614, -0.0273],
        [-0.3471, -0.5016, -0.4241,  1.3546],
        [ 0.7712, -1.4407, -1.3964,  0.5002]])

In [104]:
class MotorModel_1(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(4 , 64)
        self.l2 = nn.Linear(64 , 128)
        self.l3 = nn.Linear(128 , 64)
        self.l4 = nn.Linear(64 , 6)
        self.r = nn.ReLU()
        self.d = nn.Dropout(0.15)
    def forward(self , x):
        return self.l4(self.d(self.r(self.l3(self.d(self.r(self.l2(self.d(self.r(self.l1(x))))))))))

In [105]:
m1 = MotorModel_1().to(device)

In [106]:
loss1 = nn.BCEWithLogitsLoss()

In [107]:
optimizer = optim.Adam(m1.parameters() , lr = 0.001)

In [108]:
train_step(m1 ,500 , acc , loss1 , optimizer , x2_train , y2_train , x2_test , y2_test)

  0%|          | 0/500 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.6964  Test Loss : 0.6923  Accuracy : 0.5117 
Epoch : 2  Train Loss : 0.6876  Test Loss : 0.6825  Accuracy : 0.5833 
Epoch : 4  Train Loss : 0.6779  Test Loss : 0.6723  Accuracy : 0.6789 
Epoch : 6  Train Loss : 0.6680  Test Loss : 0.6608  Accuracy : 0.7728 
Epoch : 8  Train Loss : 0.6569  Test Loss : 0.6476  Accuracy : 0.8656 
Epoch : 10  Train Loss : 0.6444  Test Loss : 0.6322  Accuracy : 0.8917 
Epoch : 12  Train Loss : 0.6266  Test Loss : 0.6140  Accuracy : 0.9044 
Epoch : 14  Train Loss : 0.6091  Test Loss : 0.5923  Accuracy : 0.9106 
Epoch : 16  Train Loss : 0.5853  Test Loss : 0.5668  Accuracy : 0.9150 
Epoch : 18  Train Loss : 0.5595  Test Loss : 0.5374  Accuracy : 0.9183 
Epoch : 20  Train Loss : 0.5277  Test Loss : 0.5042  Accuracy : 0.9256 
Epoch : 22  Train Loss : 0.4992  Test Loss : 0.4676  Accuracy : 0.9322 
Epoch : 24  Train Loss : 0.4623  Test Loss : 0.4285  Accuracy : 0.9378 
Epoch : 26  Train Loss : 0.4234  Test Loss : 0.3880  Accuracy : 0.940

In [109]:
import os
os.makedirs("saved_models", exist_ok=True)
torch.save(m1.state_dict(), "saved_models/m1.pth")

In [110]:
class DeepResNet(nn.Module):

    def __init__(self):

        super().__init__()
        self.input = nn.Linear(4, 64)
        self.block1 = ResidualBlock(64)
        self.block2 = ResidualBlock(64)
        self.output = nn.Linear(64, 6)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.input(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x

In [111]:
m2 = DeepResNet().to(device)

In [112]:
op = optim.Adam(m2.parameters() , lr = 0.001)

In [113]:
train_step(m2 ,500 , acc , loss , op , x2_train , y2_train , x2_test , y2_test)

  0%|          | 0/500 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.7065  Test Loss : 0.6997  Accuracy : 0.5294 
Epoch : 2  Train Loss : 0.6900  Test Loss : 0.6838  Accuracy : 0.5917 
Epoch : 4  Train Loss : 0.6743  Test Loss : 0.6685  Accuracy : 0.6511 
Epoch : 6  Train Loss : 0.6591  Test Loss : 0.6532  Accuracy : 0.7106 
Epoch : 8  Train Loss : 0.6438  Test Loss : 0.6376  Accuracy : 0.7672 
Epoch : 10  Train Loss : 0.6279  Test Loss : 0.6212  Accuracy : 0.8100 
Epoch : 12  Train Loss : 0.6113  Test Loss : 0.6038  Accuracy : 0.8433 
Epoch : 14  Train Loss : 0.5936  Test Loss : 0.5850  Accuracy : 0.8611 
Epoch : 16  Train Loss : 0.5744  Test Loss : 0.5644  Accuracy : 0.8694 
Epoch : 18  Train Loss : 0.5532  Test Loss : 0.5414  Accuracy : 0.8883 
Epoch : 20  Train Loss : 0.5296  Test Loss : 0.5159  Accuracy : 0.9033 
Epoch : 22  Train Loss : 0.5033  Test Loss : 0.4877  Accuracy : 0.9172 
Epoch : 24  Train Loss : 0.4744  Test Loss : 0.4570  Accuracy : 0.9233 
Epoch : 26  Train Loss : 0.4430  Test Loss : 0.4240  Accuracy : 0.930

In [114]:
torch.save(m2.state_dict(), "saved_models/m2.pth")

In [115]:
class MLP(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.layer_stack = nn.Sequential(
            
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 6)
        )
        
    def forward(self, x):
        
        return self.layer_stack(x)


m3 = MLP()

In [116]:
op1 = optim.Adam(m3.parameters() , lr = 0.001)

In [117]:
train_step(m3 ,500 , acc , loss , op1 , x2_train , y2_train , x2_test , y2_test)

  0%|          | 0/500 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.6965  Test Loss : 0.6883  Accuracy : 0.5700 
Epoch : 2  Train Loss : 0.6870  Test Loss : 0.6796  Accuracy : 0.6211 
Epoch : 4  Train Loss : 0.6776  Test Loss : 0.6711  Accuracy : 0.6578 
Epoch : 6  Train Loss : 0.6685  Test Loss : 0.6626  Accuracy : 0.7050 
Epoch : 8  Train Loss : 0.6608  Test Loss : 0.6540  Accuracy : 0.7372 
Epoch : 10  Train Loss : 0.6523  Test Loss : 0.6450  Accuracy : 0.7739 
Epoch : 12  Train Loss : 0.6413  Test Loss : 0.6355  Accuracy : 0.8106 
Epoch : 14  Train Loss : 0.6320  Test Loss : 0.6255  Accuracy : 0.8456 
Epoch : 16  Train Loss : 0.6211  Test Loss : 0.6148  Accuracy : 0.8694 
Epoch : 18  Train Loss : 0.6089  Test Loss : 0.6033  Accuracy : 0.8922 
Epoch : 20  Train Loss : 0.5962  Test Loss : 0.5911  Accuracy : 0.9100 
Epoch : 22  Train Loss : 0.5840  Test Loss : 0.5780  Accuracy : 0.9178 
Epoch : 24  Train Loss : 0.5738  Test Loss : 0.5641  Accuracy : 0.9228 
Epoch : 26  Train Loss : 0.5566  Test Loss : 0.5493  Accuracy : 0.930

In [118]:
torch.save(m3.state_dict(), "saved_models/m3.pth")

In [119]:
class MotorANN(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        self.network = nn.Sequential(
            
            nn.Linear(4, 50),
            nn.Sigmoid(),
            
            nn.Linear(50, 50),
            nn.Sigmoid(),
            
            nn.Linear(50, 6)
        )
        
    def forward(self, x):
        
        return self.network(x)


m4 = MotorANN()

In [120]:
optimizer = torch.optim.Adam(m4.parameters(),lr=0.001)

In [121]:
train_step(m4 ,500 , acc , loss , optimizer , x2_train , y2_train , x2_test , y2_test)

  0%|          | 0/500 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.6980  Test Loss : 0.6948  Accuracy : 0.5133 
Epoch : 2  Train Loss : 0.6947  Test Loss : 0.6921  Accuracy : 0.5189 
Epoch : 4  Train Loss : 0.6922  Test Loss : 0.6900  Accuracy : 0.5411 
Epoch : 6  Train Loss : 0.6904  Test Loss : 0.6886  Accuracy : 0.5411 
Epoch : 8  Train Loss : 0.6891  Test Loss : 0.6875  Accuracy : 0.5406 
Epoch : 10  Train Loss : 0.6881  Test Loss : 0.6866  Accuracy : 0.5400 
Epoch : 12  Train Loss : 0.6871  Test Loss : 0.6856  Accuracy : 0.5406 
Epoch : 14  Train Loss : 0.6861  Test Loss : 0.6847  Accuracy : 0.5506 
Epoch : 16  Train Loss : 0.6851  Test Loss : 0.6837  Accuracy : 0.6094 
Epoch : 18  Train Loss : 0.6841  Test Loss : 0.6826  Accuracy : 0.6228 
Epoch : 20  Train Loss : 0.6830  Test Loss : 0.6815  Accuracy : 0.6100 
Epoch : 22  Train Loss : 0.6819  Test Loss : 0.6803  Accuracy : 0.6228 
Epoch : 24  Train Loss : 0.6807  Test Loss : 0.6790  Accuracy : 0.6372 
Epoch : 26  Train Loss : 0.6794  Test Loss : 0.6776  Accuracy : 0.651

In [122]:
torch.save(m4.state_dict(), "saved_models/m4.pth")

In [123]:

class ModernANN(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        self.network = nn.Sequential(
            
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 6)
        )
        
    def forward(self, x):
        
        return self.network(x)


m5 = ModernANN()


In [124]:
modern_optimizer = torch.optim.Adam(
    m5.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [125]:
train_step(m5 ,500 , acc , loss , modern_optimizer , x2_train , y2_train , x2_test , y2_test)

  0%|          | 0/500 [00:00<?, ?it/s]

Epoch : 0  Train Loss : 0.6871  Test Loss : 0.6820  Accuracy : 0.5978 
Epoch : 2  Train Loss : 0.6775  Test Loss : 0.6730  Accuracy : 0.6689 
Epoch : 4  Train Loss : 0.6686  Test Loss : 0.6638  Accuracy : 0.7200 
Epoch : 6  Train Loss : 0.6595  Test Loss : 0.6545  Accuracy : 0.7572 
Epoch : 8  Train Loss : 0.6509  Test Loss : 0.6449  Accuracy : 0.7772 
Epoch : 10  Train Loss : 0.6400  Test Loss : 0.6348  Accuracy : 0.8056 
Epoch : 12  Train Loss : 0.6311  Test Loss : 0.6241  Accuracy : 0.8294 
Epoch : 14  Train Loss : 0.6205  Test Loss : 0.6128  Accuracy : 0.8589 
Epoch : 16  Train Loss : 0.6077  Test Loss : 0.6007  Accuracy : 0.8717 
Epoch : 18  Train Loss : 0.5953  Test Loss : 0.5877  Accuracy : 0.8789 
Epoch : 20  Train Loss : 0.5830  Test Loss : 0.5739  Accuracy : 0.8878 
Epoch : 22  Train Loss : 0.5676  Test Loss : 0.5592  Accuracy : 0.8939 
Epoch : 24  Train Loss : 0.5544  Test Loss : 0.5437  Accuracy : 0.9017 
Epoch : 26  Train Loss : 0.5388  Test Loss : 0.5273  Accuracy : 0.906

In [126]:
torch.save(m5.state_dict(), "saved_models/m5.pth")